The importation

In [1]:
import requests
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import min,max,col,explode, split, desc,lit
from datetime import datetime

1. load data from here. This should be done using a notebook cell and not a manual process to import the data. For some of the datasets you may get an error that the file is too big. It is acceptable to load these manually if needed, please note files that are expected to be added manually.

In [2]:
name_list=["name.basics.tsv.gz","title.akas.tsv.gz","title.basics.tsv.gz","title.crew.tsv.gz","title.episode.tsv.gz","title.principals.tsv.gz","title.ratings.tsv.gz"]

In [3]:
output_folder = "data"
os.makedirs(output_folder, exist_ok=True)

for name in name_list:
    file_path = os.path.join(output_folder, name)
    if os.path.exists(file_path):
        print(f"The file '{file_path}' already exist. Dowload ignore.")
        continue

    url = "https://datasets.imdbws.com/"+ name
    
    response = requests.get(url, stream=True)
    
    if response.status_code == 200:
        with open(file_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        print("Download complete!")
    else:
        print("Request failed:", response.status_code)


Download complete!
Download complete!
Download complete!
Download complete!
Download complete!
Download complete!
Download complete!


We create the spark app then create dfs that contain all the dataframe for each file

In [4]:
spark = SparkSession.builder.appName("IMDB").getOrCreate()



In [5]:
dfs = {}
input_folder = "data"
for name in name_list:
    file_path = os.path.join(input_folder, name)
    df = spark.read.csv(
        file_path,
        sep="\t",
        header=True,
        quote="",
        escape="",
        inferSchema=True
    )
    # Replace  "\N" by null
    df = df.replace("\\N", None)

    dfs[name] = df

    print(f"\nDataFrame pour {name}:")
    df.show(5)


DataFrame pour name.basics.tsv.gz:
+---------+---------------+---------+---------+--------------------+--------------------+
|   nconst|    primaryName|birthYear|deathYear|   primaryProfession|      knownForTitles|
+---------+---------------+---------+---------+--------------------+--------------------+
|nm0000001|   Fred Astaire|     1899|     1987|actor,miscellaneo...|tt0072308,tt00504...|
|nm0000002|  Lauren Bacall|     1924|     2014|actress,miscellan...|tt0037382,tt00752...|
|nm0000003|Brigitte Bardot|     1934|     NULL|actress,music_dep...|tt0057345,tt00491...|
|nm0000004|   John Belushi|     1949|     1982|actor,writer,musi...|tt0072562,tt00779...|
|nm0000005| Ingmar Bergman|     1918|     2007|writer,director,a...|tt0050986,tt00694...|
+---------+---------------+---------+---------+--------------------+--------------------+
only showing top 5 rows

DataFrame pour title.akas.tsv.gz:
+---------+--------+--------------------+------+--------+-----------+-------------+------------

2. How many total people in data set?

In [6]:
df_name = dfs["name.basics.tsv.gz"].withColumn("birthYear", col("birthYear").cast("int"))

In [7]:
total_count=df_name.count()

In [8]:
total_count

14951110

In [9]:
unique_names = df_name.select("primaryName").distinct().count()

print(f"Number of primary uniques name : {unique_names}")

Number of primary uniques name : 11414365


3. What is the earliest year of birth?

In [10]:

min_birth_date = df_name.select(min("birthYear")).collect()[0][0]

In [11]:
min_birth_date

4

4. How many years ago was this person born?

In [12]:
current_year = datetime.now().year
years_ago = current_year - min_birth_date if min_birth_date else None

In [13]:
years_ago

2021

5. Using only the data in the data set, determine if this date of birth correct.

In [14]:
df_100 = df_name.filter(df_name.birthYear == min_birth_date)

# Afficher le résultat
df_100.show()


+---------+------------------+---------+---------+-----------------+--------------------+
|   nconst|       primaryName|birthYear|deathYear|primaryProfession|      knownForTitles|
+---------+------------------+---------+---------+-----------------+--------------------+
|nm0784172|Lucio Anneo Seneca|        4|       65|           writer|tt0043802,tt02188...|
+---------+------------------+---------+---------+-----------------+--------------------+



In [15]:
min_death_date = df_100.select("deathYear").collect()[0][0]

In [16]:
min_death_date

'65'

In [17]:
calculate_age= int(min_death_date)-int(min_birth_date)

In [18]:
calculate_age

61

This Date of birth is corecte, 61 years is old (for the time) but understandable

6. Explain the reasoning for the answer in a code comment or new markdown cell.

The birth date is correct because the calculated age ($65 - 4 = 61$) is a positive and biologically plausible lifespan for a human. Since the individual is a documented historical figure, this internal data consistency validates the entry within the dataset.

7. What is the most recent date of birth?

In [19]:
max_birth_date = df_name.select(max("birthYear")).collect()[0][0]

In [20]:
max_birth_date

2025

8. What percentage of the people do not have a listed date of birth?

In [21]:
no_birth_date_count = df_name.filter(col("birthYear").isNull()).count()

In [22]:
percentage_no_birth_date = (no_birth_date_count / total_count) * 100

In [23]:
percentage_no_birth_date

95.57668293524695

9. What is the length of the longest "short" after 1900?

In [24]:
df_basics = dfs["title.basics.tsv.gz"].withColumn("startYear", col("startYear").cast("int")) \
                                     .withColumn("runtimeMinutes", col("runtimeMinutes").cast("int"))

In [25]:
longest_short = df_basics.filter((col("titleType") == "short") & (col("startYear") > 1900)) \
                         .select(max("runtimeMinutes")).collect()[0][0]

In [26]:
longest_short

1311

10. What is the length of the shortest "movie" after 1900?

In [27]:


shortest_movie = df_basics.filter(
    (col("titleType") == "movie") & (col("startYear") > 1900)
).agg(min("runtimeMinutes")).collect()[0][0]

In [28]:
shortest_movie

1

11. List of all of the genres represented.

In [29]:
distinct_genres_df = df_basics.select(explode(split(col("genres"), ","))).distinct()

list_of_genres = [row[0] for row in distinct_genres_df.collect() if row[0] is not None]

In [30]:
list_of_genres

['Crime',
 'Romance',
 'Thriller',
 'Adventure',
 'Drama',
 'War',
 'Documentary',
 'Reality-TV',
 'Family',
 'Fantasy',
 'Game-Show',
 'Adult',
 'History',
 'Mystery',
 'Musical',
 'Animation',
 'Music',
 'Film-Noir',
 'Short',
 'Horror',
 'Western',
 'Biography',
 'Comedy',
 'Sport',
 'Action',
 'Talk-Show',
 'Sci-Fi',
 'News']

12. What is the highest rated comedy "movie" in the dataset? Note, if there is a tie, the tie shall be broken by the movie with the most votes .

In [31]:
df_ratings = dfs["title.ratings.tsv.gz"]

In [32]:
best_comedy = df_basics.join(df_ratings, "tconst") \
    .filter((col("titleType") == "movie") & (col("genres").contains("Comedy"))) \
    .orderBy(desc("averageRating"), desc("numVotes"))\
    .limit(1)\
    .collect()[0]


In [33]:
movie_id = best_comedy["tconst"]
movie_title = best_comedy["primaryTitle"]

In [34]:
movie_id

'tt32752452'

In [35]:
movie_title

'Space Melody'

13. Who was the director of the movie?

In [36]:
df_crew = dfs["title.crew.tsv.gz"]

In [37]:
director_nconst = df_crew.filter(col("tconst") == movie_id).select("directors").collect()[0][0]

In [38]:
director_name = df_name.filter(col("nconst") == director_nconst).select("primaryName").collect()[0][0]

In [39]:
director_name

'Leonardo Thimo'

14. List, if any, the alternate titles for the movie.

In [40]:
df_akas = dfs["title.akas.tsv.gz"]

In [41]:


alternate_titles_df = df_akas.filter(col("titleId") == movie_id)

alternate_titles = [row["title"] for row in alternate_titles_df.select("title").distinct().collect()]

In [42]:
alternate_titles

['H Melwdia Tou Diastimatos', 'Space Melody', "Leonardo Thimo's Space Melody"]